In [1]:
!nvidia-smi

Sat Jul 11 19:06:37 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import torch.optim as optim

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [4]:
transform = transforms.ToTensor()

#Load MNIST data
train_dataset = datasets.MNIST(
    root="./data",
    train = True,
    download=True,
    transform=transform
)


test_dataset = datasets.MNIST(
    root="./data",
    train = False,
    download=True,
    transform=transform
)

100%|██████████| 9.91M/9.91M [00:00<00:00, 17.9MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 481kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.47MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 16.4MB/s]


In [5]:
# Hyperparameters
batch_size = 64
learning_rate = 0.001
epochs = 5

In [6]:
train_loader = DataLoader(
    train_dataset,
    batch_size = batch_size,
    shuffle= True
)

test_loader = DataLoader(
    test_dataset,
    batch_size = batch_size,
    shuffle= False
)

In [7]:
# Neural Networks
class SimpleANN(nn.Module):
  def __init__(self):
    super().__init__()

    self.fc1 = nn.Linear(28 * 28, 128)
    self.relu = nn.ReLU()
    self.fc2 = nn.Linear(128, 10)

  def forward(self, x):
      x = x.view(x.size(0), -1)  # Flatten
      x = self.relu(self.fc1(x))
      x = self.fc2(x)
      return x

In [8]:
model = SimpleANN().to(device)

In [9]:
model

SimpleANN(
  (fc1): Linear(in_features=784, out_features=128, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=128, out_features=10, bias=True)
)

In [10]:
# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

In [11]:
# Training loop
for epoch in range(epochs):
    model.train()
    running_loss = 0.0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    avg_loss = running_loss / len(train_loader)
    print(f"Epoch [{epoch+1}/{epochs}], Loss: {avg_loss:.4f}")


Epoch [1/5], Loss: 0.3473
Epoch [2/5], Loss: 0.1584
Epoch [3/5], Loss: 0.1098
Epoch [4/5], Loss: 0.0820
Epoch [5/5], Loss: 0.0657


In [12]:
# Evaluation
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f"Test Accuracy: {accuracy:.2f}%")

Test Accuracy: 97.68%


In [13]:
torch.save(model.state_dict(), "mnist_model.pth")

# Prediction

In [14]:
from PIL import Image

In [15]:
transform = transforms.Compose([
    transforms.Grayscale(),
    transforms.Resize((28, 28)),
    transforms.ToTensor(),
])

In [17]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

image = Image.open("/content/images.png")

x = transform(image)
x = x.unsqueeze(0)
x = x.to(device)  # Move input to same device as model

model = model.to(device)

with torch.no_grad():
    output = model(x)
    prediction = output.argmax(dim=1).item()

print("Predicted digit:", prediction)

Predicted digit: 7
